# 07 — Ablation Studies
Runs 6 experiments and saves `metrics/ablation_results.json`.

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import os
import json, warnings, shutil, subprocess, tempfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import DistilBertTokenizerFast

# Reduce native runtime conflicts (OpenMP/BLAS) in mixed ML stacks.
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

from src.text_pipeline import TextEncoder
from src.fusion_model import HybridSalesPredictor, GatedFusion

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODELS_DIR = ROOT / 'models'
METRICS_DIR = ROOT / 'metrics'
FIGURES_DIR = ROOT / 'figures'
SAMPLE_SIZE = 3000
BATCH_SIZE = 32
EPOCHS = 3
MAX_LEN = 128
REPR_DIM = 128
print(f'Device={DEVICE}, Sample={SAMPLE_SIZE}')

Device=cpu, Sample=3000


## Load & prepare data (shared across all experiments)

In [2]:
ds = load_dataset('DeepMostInnovations/saas-sales-conversations', split='train')
df = ds.to_pandas().sample(SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
y  = df['outcome'].astype(int).values

# Tabular
num_cols = [c for c in ['conversation_length','customer_engagement','sales_effectiveness'] if c in df.columns]
cat_cols = [c for c in ['product_type','conversation_style','communication_channel'] if c in df.columns]
tab_df = df[num_cols + cat_cols].copy()
if 'customer_engagement' in tab_df.columns and 'sales_effectiveness' in tab_df.columns:
    tab_df['eng_eff_ratio'] = tab_df['customer_engagement'] / (tab_df['sales_effectiveness'] + 0.001)
fin_num = [c for c in tab_df.columns if c not in cat_cols]
scaler = StandardScaler()
X_num = scaler.fit_transform(tab_df[fin_num].fillna(0).values)
if cat_cols:
    enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_cat = enc.fit_transform(tab_df[cat_cols].fillna('unknown').astype(str))
    X_tab = np.hstack([X_num, X_cat])
else:
    X_tab = X_num

# XGBoost + leaf encoding (run in isolated subprocess to prevent kernel crashes)
X_tab = np.ascontiguousarray(np.asarray(X_tab, dtype=np.float32))
y = np.asarray(y, dtype=np.int32)

tmp_dir = Path(tempfile.mkdtemp(prefix='ablation_xgb_'))
data_path = tmp_dir / 'train.npz'
leaf_path = tmp_dir / 'leaf_idx.npy'
model_path = MODELS_DIR / 'xgboost_model.json'
np.savez_compressed(data_path, X=X_tab, y=y)

subprocess_script = r"""
import os
import sys
from pathlib import Path
import numpy as np
import xgboost as xgb

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

data_path = Path(sys.argv[1])
model_path = Path(sys.argv[2])
leaf_path = Path(sys.argv[3])
seed = int(sys.argv[4])

data = np.load(data_path)
X = np.ascontiguousarray(data["X"], dtype=np.float32)
y = np.asarray(data["y"], dtype=np.int32)

clf = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    eval_metric="logloss",
    random_state=seed,
    verbosity=0,
    n_jobs=1,
    tree_method="hist",
)
clf.fit(X, y)
clf.save_model(str(model_path))
leaf_idx = clf.get_booster().predict(xgb.DMatrix(X), pred_leaf=True)
np.save(leaf_path, leaf_idx)
"""

used_backend = None
try:
    res = subprocess.run(
        [
            sys.executable,
            '-c',
            subprocess_script,
            str(data_path),
            str(model_path),
            str(leaf_path),
            str(SEED),
        ],
        check=False,
        capture_output=True,
        text=True,
    )

    if res.returncode == 0 and leaf_path.exists() and model_path.exists():
        leaf_idx = np.load(leaf_path)
        used_backend = 'xgboost'
        print('XGBoost leaf encoding created in subprocess.')
    else:
        print('XGBoost subprocess failed; falling back to RandomForest leaf encoding.')
        if res.stdout:
            print(res.stdout)
        if res.stderr:
            print(res.stderr)

        model_path.unlink(missing_ok=True)
        rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=1)
        rf.fit(X_tab, y)
        leaf_idx = rf.apply(X_tab)
        used_backend = 'sklearn_rf'
        print(f'RandomForest leaf encoding created: {leaf_idx.shape}')
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

if used_backend is None:
    raise RuntimeError('Failed to create tabular leaf encoding.')

leaf_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
leaf_ohe = leaf_enc.fit_transform(leaf_idx)
LEAF_DIM = leaf_ohe.shape[1]

# Text tokenisation
text_col = 'full_text' if 'full_text' in df.columns else 'conversation'
texts = df[text_col].fillna('').astype(str).tolist()
tok = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
enc_out = tok(texts, max_length=MAX_LEN, padding='max_length', truncation=True, return_tensors='pt')
input_ids  = enc_out['input_ids']
attn_masks = enc_out['attention_mask']

# Edge-case mask: ambiguous tabular signals
edge_mask = np.ones(len(y), dtype=bool)
if 'customer_engagement' in df.columns:
    edge_mask &= (df['customer_engagement'].between(0.3, 0.7).values)
if 'sales_effectiveness' in df.columns:
    edge_mask &= (df['sales_effectiveness'].between(0.3, 0.7).values)
if 'conversation_length' in df.columns:
    edge_mask &= (df['conversation_length'].between(5, 15).values)
print(f'Edge-case subset: {edge_mask.sum()} / {len(y)}')

# Train/val split (shared indices)
idx = np.arange(len(y))
tr_idx, va_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=SEED)

leaf_t = torch.tensor(leaf_ohe, dtype=torch.float32)
tab_t  = torch.tensor(X_tab, dtype=torch.float32)
y_t    = torch.tensor(y, dtype=torch.float32)

def make_loader(indices, tensors, shuffle):
    ds = TensorDataset(*[t[indices] for t in tensors])
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

print('Data ready.')

XGBoost leaf encoding created in subprocess.
Edge-case subset: 1766 / 3000
Data ready.


## Training helper

In [3]:
def train_and_eval(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3):
    criterion = nn.BCELoss()
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    model = model.to(DEVICE)
    for ep in range(epochs):
        model.train()
        for batch in train_loader:
            inputs = [b.to(DEVICE) for b in batch[:-1]]
            yb = batch[-1].to(DEVICE)
            out = model(*inputs)
            prob = out[0].squeeze(-1) if isinstance(out, tuple) else out.squeeze(-1)
            loss = criterion(prob, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    model.eval()
    probs_all, preds_all, labels_all = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            inputs = [b.to(DEVICE) for b in batch[:-1]]
            yb = batch[-1]
            out = model(*inputs)
            prob = out[0].squeeze(-1) if isinstance(out, tuple) else out.squeeze(-1)
            prob = prob.cpu()
            probs_all.extend(prob.numpy())
            preds_all.extend((prob.numpy() > 0.5).astype(int))
            labels_all.extend(yb.numpy())
    probs_all = np.array(probs_all)
    preds_all = np.array(preds_all)
    labels_all = np.array(labels_all)
    f1  = f1_score(labels_all, preds_all, zero_division=0)
    auc = roc_auc_score(labels_all, probs_all) if len(np.unique(labels_all))>1 else 0.0
    acc = accuracy_score(labels_all, preds_all)
    return f1, auc, acc, probs_all, preds_all, labels_all

## Experiment A — Full Hybrid

In [4]:
print('=== Exp A: Full Hybrid ===')
text_enc_a = TextEncoder(output_dim=REPR_DIM, freeze_bert=True)
model_a = HybridSalesPredictor(text_encoder=text_enc_a, repr_dim=REPR_DIM)
model_a.set_tab_projection(leaf_dim=LEAF_DIM)

tr_a = make_loader(tr_idx, [leaf_t, input_ids, attn_masks, y_t], True)
va_a = make_loader(va_idx, [leaf_t, input_ids, attn_masks, y_t], False)

f1_a, auc_a, acc_a, prob_a, pred_a, lab_a = train_and_eval(model_a, tr_a, va_a)

# Edge-case F1 on val set
va_edge = edge_mask[va_idx]
ef1_a = f1_score(lab_a[va_edge], pred_a[va_edge], zero_division=0) if va_edge.sum() > 0 else 0.0
print(f'  F1={f1_a:.4f} AUC={auc_a:.4f} Acc={acc_a:.4f} Edge-F1={ef1_a:.4f}')

=== Exp A: Full Hybrid ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1=0.9761 AUC=0.9981 Acc=0.9767 Edge-F1=0.8627


## Experiment B — Tabular Only (XGBoost baseline)

In [6]:
print('=== Exp B: Tabular Only (XGBoost) ===')
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000, random_state=SEED)
lr_model.fit(X_tab[tr_idx], y[tr_idx])
pred_b = lr_model.predict(X_tab[va_idx])
prob_b = lr_model.predict_proba(X_tab[va_idx])[:, 1]
lab_b  = y[va_idx]
f1_b   = f1_score(lab_b, pred_b, zero_division=0)
auc_b  = roc_auc_score(lab_b, prob_b) if len(np.unique(lab_b))>1 else 0.0
acc_b  = accuracy_score(lab_b, pred_b)
ef1_b  = f1_score(lab_b[va_edge], pred_b[va_edge], zero_division=0) if va_edge.sum()>0 else 0.0
print(f'  F1={f1_b:.4f} AUC={auc_b:.4f} Acc={acc_b:.4f} Edge-F1={ef1_b:.4f}')

=== Exp B: Tabular Only (XGBoost) ===
  F1=0.9732 AUC=0.9975 Acc=0.9733 Edge-F1=0.8596


## Experiment C — Text Only

In [7]:
print('=== Exp C: Text Only ===')

class TextOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = TextEncoder(output_dim=REPR_DIM, freeze_bert=True)
        self.head = nn.Sequential(nn.Linear(REPR_DIM,64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64,1), nn.Sigmoid())
    def forward(self, input_ids, attention_mask):
        text_repr, _ = self.encoder(input_ids, attention_mask)
        return self.head(text_repr), None

model_c = TextOnlyModel()
tr_c = make_loader(tr_idx, [input_ids, attn_masks, y_t], True)
va_c = make_loader(va_idx, [input_ids, attn_masks, y_t], False)
f1_c, auc_c, acc_c, prob_c, pred_c, lab_c = train_and_eval(model_c, tr_c, va_c)
ef1_c = f1_score(lab_c[va_edge], pred_c[va_edge], zero_division=0) if va_edge.sum()>0 else 0.0
print(f'  F1={f1_c:.4f} AUC={auc_c:.4f} Acc={acc_c:.4f} Edge-F1={ef1_c:.4f}')

=== Exp C: Text Only ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1=0.6203 AUC=0.6232 Acc=0.5817 Edge-F1=0.3580


## Experiment D — Concatenation (no gate)

In [8]:
print('=== Exp D: Concat (no gate) ===')

class ConcatModel(nn.Module):
    def __init__(self, leaf_dim):
        super().__init__()
        self.tab_proj = nn.Sequential(nn.Linear(leaf_dim, REPR_DIM), nn.LayerNorm(REPR_DIM), nn.ReLU())
        self.txt_enc  = TextEncoder(output_dim=REPR_DIM, freeze_bert=True)
        self.head = nn.Sequential(nn.Linear(REPR_DIM*2,128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128,1), nn.Sigmoid())
    def forward(self, leaf_feat, input_ids, attention_mask):
        tab = self.tab_proj(leaf_feat)
        txt, _ = self.txt_enc(input_ids, attention_mask)
        return self.head(torch.cat([tab, txt], dim=1)), None

model_d = ConcatModel(LEAF_DIM)
tr_d = make_loader(tr_idx, [leaf_t, input_ids, attn_masks, y_t], True)
va_d = make_loader(va_idx, [leaf_t, input_ids, attn_masks, y_t], False)
f1_d, auc_d, acc_d, prob_d, pred_d, lab_d = train_and_eval(model_d, tr_d, va_d)
ef1_d = f1_score(lab_d[va_edge], pred_d[va_edge], zero_division=0) if va_edge.sum()>0 else 0.0
print(f'  F1={f1_d:.4f} AUC={auc_d:.4f} Acc={acc_d:.4f} Edge-F1={ef1_d:.4f}')

=== Exp D: Concat (no gate) ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1=0.9688 AUC=0.9987 Acc=0.9683 Edge-F1=0.8480


## Experiment E — Frozen BERT (no fine-tune)

In [9]:
print('=== Exp E: Frozen BERT ===')
text_enc_e = TextEncoder(output_dim=REPR_DIM, freeze_bert=True)
model_e = HybridSalesPredictor(text_encoder=text_enc_e, repr_dim=REPR_DIM)
model_e.set_tab_projection(leaf_dim=LEAF_DIM)
# No phase-2 unfreeze — stays frozen
tr_e = make_loader(tr_idx, [leaf_t, input_ids, attn_masks, y_t], True)
va_e = make_loader(va_idx, [leaf_t, input_ids, attn_masks, y_t], False)
f1_e, auc_e, acc_e, prob_e, pred_e, lab_e = train_and_eval(model_e, tr_e, va_e, epochs=EPOCHS+2)
ef1_e = f1_score(lab_e[va_edge], pred_e[va_edge], zero_division=0) if va_edge.sum()>0 else 0.0
print(f'  F1={f1_e:.4f} AUC={auc_e:.4f} Acc={acc_e:.4f} Edge-F1={ef1_e:.4f}')

=== Exp E: Frozen BERT ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1=0.9747 AUC=0.9986 Acc=0.9750 Edge-F1=0.8624


## Experiment F — No XGB Encoding (raw features)

In [10]:
print('=== Exp F: No XGB Encoding (raw tab features) ===')
RAW_DIM = X_tab.shape[1]
raw_t = torch.tensor(X_tab, dtype=torch.float32)

text_enc_f = TextEncoder(output_dim=REPR_DIM, freeze_bert=True)
model_f = HybridSalesPredictor(text_encoder=text_enc_f, repr_dim=REPR_DIM)
model_f.set_tab_projection(leaf_dim=RAW_DIM)  # raw features instead of leaf OHE

tr_f = make_loader(tr_idx, [raw_t, input_ids, attn_masks, y_t], True)
va_f = make_loader(va_idx, [raw_t, input_ids, attn_masks, y_t], False)
f1_f, auc_f, acc_f, prob_f, pred_f, lab_f = train_and_eval(model_f, tr_f, va_f)
ef1_f = f1_score(lab_f[va_edge], pred_f[va_edge], zero_division=0) if va_edge.sum()>0 else 0.0
print(f'  F1={f1_f:.4f} AUC={auc_f:.4f} Acc={acc_f:.4f} Edge-F1={ef1_f:.4f}')

=== Exp F: No XGB Encoding (raw tab features) ===


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1=0.9593 AUC=0.9964 Acc=0.9600 Edge-F1=0.7736


## Compile results & save

In [11]:
ablation = {
    'A_full_hybrid':   {'f1': f1_a,  'auc': auc_a,  'acc': acc_a,  'edge_case_f1': ef1_a},
    'B_tabular_only':  {'f1': f1_b,  'auc': auc_b,  'acc': acc_b,  'edge_case_f1': ef1_b},
    'C_text_only':     {'f1': f1_c,  'auc': auc_c,  'acc': acc_c,  'edge_case_f1': ef1_c},
    'D_concat_no_gate':{'f1': f1_d,  'auc': auc_d,  'acc': acc_d,  'edge_case_f1': ef1_d},
    'E_frozen_text':   {'f1': f1_e,  'auc': auc_e,  'acc': acc_e,  'edge_case_f1': ef1_e},
    'F_no_xgb_encode': {'f1': f1_f,  'auc': auc_f,  'acc': acc_f,  'edge_case_f1': ef1_f},
}

with open(METRICS_DIR / 'ablation_results.json', 'w') as f:
    json.dump(ablation, f, indent=2)

print('\n=== ABLATION SUMMARY ===')
print(f'{"Model":<22} {"F1":>6} {"AUC":>6} {"Acc":>6} {"Edge-F1":>8}')
print('-'*52)
for name, m in ablation.items():
    print(f'{name:<22} {m["f1"]:>6.4f} {m["auc"]:>6.4f} {m["acc"]:>6.4f} {m["edge_case_f1"]:>8.4f}')


=== ABLATION SUMMARY ===
Model                      F1    AUC    Acc  Edge-F1
----------------------------------------------------
A_full_hybrid          0.9761 0.9981 0.9767   0.8627
B_tabular_only         0.9732 0.9975 0.9733   0.8596
C_text_only            0.6203 0.6232 0.5817   0.3580
D_concat_no_gate       0.9688 0.9987 0.9683   0.8480
E_frozen_text          0.9747 0.9986 0.9750   0.8624
F_no_xgb_encode        0.9593 0.9964 0.9600   0.7736


## Plots

In [12]:
names = [k.replace('_',' ') for k in ablation]
f1s   = [v['f1'] for v in ablation.values()]
ef1s  = [v['edge_case_f1'] for v in ablation.values()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall F1 bar chart
bars = axes[0].barh(names, f1s, color='steelblue')
axes[0].set_xlabel('F1 Score'); axes[0].set_title('Ablation — Overall F1')
axes[0].set_xlim(0, 1)
for bar, v in zip(bars, f1s):
    axes[0].text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=8)

# Overall vs Edge-case grouped
x = np.arange(len(names))
w = 0.35
axes[1].bar(x-w/2, f1s, w, label='Overall F1', color='steelblue')
axes[1].bar(x+w/2, ef1s, w, label='Edge-Case F1', color='coral')
axes[1].set_xticks(x); axes[1].set_xticklabels(names, rotation=30, ha='right', fontsize=7)
axes[1].set_title('Overall vs Edge-Case F1'); axes[1].legend(); axes[1].set_ylim(0,1)

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / 'ablation_comparison.png'), dpi=150)
plt.show()
print('Saved ablation_comparison.png. Done ✅')

Saved ablation_comparison.png. Done ✅
